In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

if groq_key:
    print(f"Groq API key loaded: {groq_key[:8]}...{groq_key[-4:]}")
else:
    print(f"Groq API key NOT found — check your .env file")


Groq API key loaded: gsk_jKpd...Kvja


In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/aws_unit_2.pdf")
pages = loader.load()

print(f"✅ Total pages loaded: {len(pages)}")
print(f"\n--- Page 1 Preview ---")
print(pages[0].page_content[:500])
print(f"\n--- Page 1 Metadata ---")
print(pages[0].metadata)

c:\Users\shubh\Documents\rag-document-qa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Total pages loaded: 97

--- Page 1 Preview ---
AWS Solution Architect
(UNIT-2:Compute & Storage)
Dr. Shagun Sharma
Assistant Professor
School of Computing Science and Engineering
shagunsharma@vitbhopal.ac.in
1

--- Page 1 Metadata ---
{'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-03-15T04:19:45+00:00', 'moddate': '2026-03-15T04:19:46+00:00', 'source': '../data/aws_unit_2.pdf', 'total_pages': 97, 'page': 0, 'page_label': '1'}


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20
)

chunks = splitter.split_documents(pages)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"📄 Total pages: {len(pages)}")
print(f"\n--- Chunk 1 Content ---")
print(chunks[0].page_content)
print(f"\n--- Chunk 1 Metadata ---")
print(chunks[0].metadata)

✅ Total chunks created: 474
📄 Total pages: 97

--- Chunk 1 Content ---
AWS Solution Architect
(UNIT-2:Compute & Storage)
Dr. Shagun Sharma
Assistant Professor
School of Computing Science and Engineering
shagunsharma@vitbhopal.ac.in
1

--- Chunk 1 Metadata ---
{'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-03-15T04:19:45+00:00', 'moddate': '2026-03-15T04:19:46+00:00', 'source': '../data/aws_unit_2.pdf', 'total_pages': 97, 'page': 0, 'page_label': '1'}


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

CHROMA_PATH = "../chroma_db"
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

if os.path.exists(CHROMA_PATH) and os.listdir(CHROMA_PATH):
    vectorstore = Chroma(
        persist_directory = CHROMA_PATH,
        embedding_function = embeddings
    )
    print(f"Loaded Total Vectors: {vectorstore._collection.count()}")
else:
    print(f"No existing vector store found. Creating new one.")
    vectorstore = Chroma.from_documents(
        documents = chunks,
        embedding = embeddings,
        persist_directory = CHROMA_PATH
    )
    print(f" Created Total Vectors: {vectorstore._collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9527.88it/s]


Loaded Total Vectors: 474


In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model_name = "llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

prompt = PromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer, say "I don't know".

Context: {context}

Question: {question}

Answer:
""")

retriever = vectorstore.as_retriever(search_kwargs={"k":4})

chain = (
    {"context": retriever, "question":RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)

question = "What is Amazon EC2?"
answer = chain.invoke(question)

print(f"Question: {question}")
print(f"\n Answer: {answer}")

Question: What is Amazon EC2?

 Answer: Amazon EC2 (Elastic Compute Cloud) is a web service provided by the AWS cloud that is secure, resizable, and scalable. It is like renting a powerful, virtual computer in the cloud that you can use to run any application.


In [6]:
docs = retriever.invoke(question)

print(f"📄 Source chunks used to answer: '{question}'\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content preview: {doc.page_content[:200]}")
    print()

📄 Source chunks used to answer: 'What is Amazon EC2?'

--- Chunk 1 ---
Page: 3
Content preview: Amazon EC2 (Elastic Compute Cloud)
• Amazon Elastic Compute Cloud (Amazon EC2) is a web service which
is provided by the AWS cloud which is secure, resizable, and scalable.

--- Chunk 2 ---
Page: 12
Content preview: • It allows its users to choose from various software present to run on their EC2 
machines. 
• This whole service is allocated to AWS Marketplace on the AWS platform.

--- Chunk 3 ---
Page: 2
Content preview: Amazon EC2 (Elastic Compute Cloud)
• Amazon EC2 is like renting a powerful, virtual computer in the cloud
that you can use to run any application.

--- Chunk 4 ---
Page: 77
Content preview: Amazon EBS (Elastic Block Store)
• EC2 is a virtual server in a cloud while EBS is a virtual disk in a cloud.
• Amazon EBS allows you to create storage volumes and attach them to 
the EC2 instances.



In [7]:
# Cell 7 — Conversation Memory (fixed v2)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the following context to answer.\n\nContext: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

def get_context(input_dict):
    docs = retriever.invoke(input_dict["question"])
    return {
        "context": "\n\n".join([d.page_content for d in docs]),
        "question": input_dict["question"],
        "chat_history": input_dict.get("chat_history", [])
    }

base_chain = RunnableLambda(get_context) | prompt | llm | StrOutputParser()

chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

print("✅ Conversational chain with memory ready!")

✅ Conversational chain with memory ready!


c:\Users\shubh\Documents\rag-document-qa\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
session_id = "test_session_1"

q1 = "What is Amazon EC2?"
a1 = chain_with_history.invoke(
    {"question": q1},
    config = {"configurable": {"session_id": session_id}}
)
print(f"Q1: {q1}")
print(f"A1: {a1}\n")

q2 = "What are its pricing models?"
a2 = chain_with_history.invoke(
    {"question": q2},
    config = {"configurable": {"session_id": session_id}}
)
print(f"Q2: {q2}")
print(f"A2: {a2}")

Q1: What is Amazon EC2?
A1: Amazon EC2 (Elastic Compute Cloud) is a web service provided by AWS that allows users to rent a powerful, virtual computer in the cloud. It is a secure, resizable, and scalable service that enables users to run any application. With EC2, users can choose from various software to run on their virtual machines, and the service is allocated to the AWS Marketplace on the AWS platform. Essentially, it's like having a virtual server in the cloud that can be used to run applications.

Q2: What are its pricing models?
A2: Amazon EC2 offers four pricing models: 

1. On-Demand Instances
2. Reserved Instances
3. Spot Instances
4. Dedicated Hosts

Additionally, Amazon EC2 also has a per-second billing scheme, which allows customers to pay only for the compute time they use, making it a cost-effective option for short-term or intermittent workloads.
